In [289]:
from bs4 import BeautifulSoup
import requests
from datetime import datetime
from zoneinfo import ZoneInfo
from datetime import datetime, timedelta
import pandas as pd

In [290]:
contests=[]
atcoder_page=requests.get("https://atcoder.jp/home").text
atcoder_soup=BeautifulSoup(atcoder_page,'lxml')

In [291]:
div_tag=atcoder_soup.find_all('div',id='contest-table-upcoming')

In [292]:
tr_tags=div_tag[0].find_all('tr')
tr_tags.pop(0)
atcoder_contests=[]

for tr in tr_tags:
    td=tr.find_all('td')
    time=td[0].find('time').text.strip()
    contest_name=td[1].find('a').text.strip()
    link ="https://atcoder.jp" + td[1].find('a')['href']
    
    
    dt = datetime.strptime(time, "%Y-%m-%d %H:%M:%S%z")
    dt = dt - timedelta(hours=3, minutes=30)

    date_time = dt.strftime("%Y-%m-%d %H:%M:%S")

    atcoder_contests.append([date_time,contest_name,link])




In [293]:
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get("https://www.codechef.com/api/list/contests/all", headers=headers)
data = response.json()

codechef_contests = []

for contest in data["future_contests"]:

    contest_code = contest["contest_code"]
    contest_name = contest["contest_name"]
    date_time = datetime.fromisoformat(contest["contest_start_date_iso"]).strftime("%Y-%m-%d %H:%M:%S")

    duration = contest["contest_duration"]

    link = "https://www.codechef.com/" + contest_code

    codechef_contests.append([
      #  contest_code,
        date_time,
        contest_name,
      #  duration,
        link
    ])

In [294]:
# -----------------------------
# LeetCode
# -----------------------------
leetcode_contest=[]
query = """
query {
    allContests {
        title
        titleSlug
        startTime
        duration
    }
}
"""

response = requests.post(
    "https://leetcode.com/graphql",
    json={"query": query},
    headers={
        "Content-Type": "application/json",
        "Referer": "https://leetcode.com/contest/"
    }
)

data = response.json()

# current unix timestamp to get only upcoming contest by comparing the ((contest timestamp and current timestamp))
now = datetime.now().timestamp()


for contest in data["data"]["allContests"]:

    if contest["startTime"] <= now:
        continue

    contest_name = contest["title"]
    date_time = datetime.fromtimestamp(contest["startTime"]).strftime("%Y-%m-%d %H:%M:%S")
    link = ("https://leetcode.com/contest/"+contest["titleSlug"])

    leetcode_contest.append([
        date_time,
        contest_name,
        link
    ])

In [295]:
# codeforces

codeforces_page = requests.get("https://codeforces.com/api/contest.list")

codeforces_contest = codeforces_page.json()
codeforces_contest = codeforces_contest["result"]

temp = []

for contest in codeforces_contest:

    if contest["relativeTimeSeconds"] < 0:

        date_time = datetime.fromtimestamp(contest["startTimeSeconds"],ZoneInfo("Asia/Kolkata")).strftime("%Y-%m-%d %H:%M:%S")
        contest_name = contest["name"]
        link = "https://codeforces.com/contest/" + str(contest["id"])

        temp.append([
            date_time,
            contest_name,
            link
        ])

codeforces_contest = temp

In [296]:
contests = (
    atcoder_contests
    + codechef_contests
    + leetcode_contest
    + codeforces_contest
)
df=pd.DataFrame(contests,columns=["date_time","contest_name","link"])
df

,date_time,contest_name,link
0,2026-09-27 17:30:00,AtCoder Grand Contest 078,https://atcoder.jp/contests/agc078
1,2026-10-03 17:30:00,AtCoder Beginner Contest 478,https://atcoder.jp/contests/abc478
2,2026-10-04 17:30:00,AtCoder Regular Contest 231,https://atcoder.jp/contests/arc231
3,2026-10-10 17:30:00,AtCoder Regular Contest++ 232,https://atcoder.jp/contests/arc232
4,2026-10-11 17:30:00,AtCoder Beginner Contest 479,https://atcoder.jp/contests/abc479
5,2026-10-12 09:30:00,AAL Contest 001: Let's use segtree!,https://atcoder.jp/contests/aalc001
6,2026-09-28 19:00:00,Monday Munch - DSA Challenge 022 (Rated),https://www.codechef.com/DSAMONDAY022
7,2026-09-30 20:00:00,Starters 258,https://www.codechef.com/START258
8,2026-10-10 20:00:00,Biweekly Contest 193,https://leetcode.com/contest/biweekly-contest-193
9,2026-10-04 08:00:00,Weekly Contest 522,https://leetcode.com/contest/weekly-contest-522
